# 03. PINN Training Monitor

**v6 Section 17.10.1** | Phase C Development Notebook #3

## Purpose
GPU 장시간 학습 모니터링 (`scripts/train_phase_c.py` 실행 중)

## Usage
1. `scripts/train_phase_c.py`가 백그라운드 실행 중일 때 이 노트북을 열어 상태 확인
2. 각 셀을 반복 실행하여 실시간 모니터링 (매 30분 갱신 권장)
3. Red flag 발생 시 학습 중단 여부 판단

## Reference
- v6 Section 7: Curriculum 3-Stage
- v6 Section 18.5: Red Flag 자동 감지

In [ ]:
import sys, json, glob
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from backend.core.pinn_model import PurePINN
from backend.training.loss_functions import ASMIncidentLUT
from backend.training.red_flag_detector import detect_red_flags

DEVICE = torch.device('cpu')  # monitor on CPU
print(f'Project: {PROJECT_ROOT.name}')

---
## Cell 1: Find Latest Experiment

In [ ]:
exp_base = PROJECT_ROOT / 'experiments'
exp_dirs = sorted(exp_base.glob('*_phase_c*'), key=lambda p: p.stat().st_mtime, reverse=True)

if not exp_dirs:
    print('No experiments found. Run scripts/train_phase_c.py first.')
else:
    EXP_DIR = exp_dirs[0]
    print(f'Latest experiment: {EXP_DIR.name}')
    print(f'Files:')
    for f in sorted(EXP_DIR.iterdir()):
        size = f.stat().st_size
        print(f'  {f.name:30s} {size/1024:.1f} KB')

---
## Cell 2: Loss History Visualization

In [ ]:
loss_path = EXP_DIR / 'loss_history.json'
if loss_path.exists():
    with open(loss_path) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(len(history['total']))
    
    # All losses
    ax = axes[0]
    ax.semilogy(epochs, history['total'], 'k-', label='Total', linewidth=1.5)
    ax.semilogy(epochs, history['L_H'], 'b-', label='L_H', alpha=0.7)
    ax.semilogy(epochs, history['L_phase'], 'r-', label='L_phase', alpha=0.7)
    ax.semilogy(epochs, history['L_BC'], 'g-', label='L_BC', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (log)')
    ax.set_title(f'Loss Curves ({len(epochs)} epochs)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Recent 20% zoom
    ax = axes[1]
    start = max(0, int(len(epochs) * 0.8))
    ax.plot(list(epochs)[start:], history['total'][start:], 'k-', label='Total', linewidth=1.5)
    ax.plot(list(epochs)[start:], history['L_H'][start:], 'b-', label='L_H', alpha=0.7)
    ax.plot(list(epochs)[start:], history['L_phase'][start:], 'r-', label='L_phase', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Recent 20% (linear scale)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'Final losses: Total={history["total"][-1]:.4e} '
          f'H={history["L_H"][-1]:.4e} Ph={history["L_phase"][-1]:.4e} '
          f'BC={history["L_BC"][-1]:.4e}')
else:
    print('Training still running or loss_history.json not yet saved.')
    print('Check training log instead:')
    log_path = EXP_DIR / 'training.log'
    if log_path.exists():
        lines = log_path.read_text().strip().split('\n')
        print('\n'.join(lines[-15:]))

---
## Cell 3: Load Latest Checkpoint & Visualize

In [ ]:
# Find latest checkpoint
ckpt_dir = PROJECT_ROOT / 'checkpoints'
ckpts = sorted(ckpt_dir.glob('phase_c_*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)

if not ckpts:
    print('No checkpoints found yet.')
else:
    latest_ckpt = ckpts[0]
    print(f'Loading: {latest_ckpt.name}')
    
    # Load config to get model dims
    cfg_path = EXP_DIR / 'config.yaml'
    if cfg_path.exists():
        import yaml
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        mcfg = cfg['model']
    else:
        mcfg = {'hidden_dim': 128, 'num_layers': 4, 'num_freqs': 48, 'omega_0': 30.0}
    
    model = PurePINN(**mcfg).to(DEVICE)
    ckpt = torch.load(latest_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f'Loaded epoch {ckpt["epoch"]}')
    
    # Visualize cross-sections
    N = 1000
    x_vis = torch.linspace(0, 504, N)
    zeros = torch.zeros(N)
    ones = torch.ones(N)
    w_def = torch.full((N,), 10.0)
    
    def eval_at_z(z_val):
        with torch.no_grad():
            coords = torch.stack([x_vis, torch.full((N,), z_val),
                                  zeros, zeros, w_def, w_def, zeros, ones], dim=1)
            U = model(coords)
            return torch.sqrt(U[:,0]**2 + U[:,1]**2).numpy()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    x_np = x_vis.numpy()
    
    for ax, (z_val, title) in zip(axes.flat, [(40, 'z=40 (BM2 boundary)'),
                                               (30, 'z=30 (ILD mid)'),
                                               (20, 'z=20 (BM1 boundary)'),
                                               (10, 'z=10 (Encap mid)')]):
        amp = eval_at_z(z_val)
        ax.plot(x_np, amp, linewidth=0.8)
        ax.set_title(f'{title}  |  std={np.std(amp):.4f}')
        ax.set_xlabel('x (μm)')
        ax.set_ylabel('|U|')
        ax.grid(True, alpha=0.3)
        for i in range(7):
            ax.axvline(x=i*72+36, color='gray', linestyle=':', alpha=0.2)
    
    plt.suptitle(f'Checkpoint: {latest_ckpt.name} (epoch {ckpt["epoch"]})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Cell 4: 2D Field Map

In [ ]:
if ckpts:
    z_range = torch.linspace(0, 40, 100)
    x_range = torch.linspace(0, 504, 200)
    Z, X = torch.meshgrid(z_range, x_range, indexing='ij')
    N_map = Z.numel()
    
    with torch.no_grad():
        coords_map = torch.stack([
            X.flatten(), Z.flatten(),
            torch.zeros(N_map), torch.zeros(N_map),
            torch.full((N_map,), 10.0), torch.full((N_map,), 10.0),
            torch.zeros(N_map), torch.ones(N_map),
        ], dim=1)
        U_map = model(coords_map)
        amp_map = torch.sqrt(U_map[:,0]**2 + U_map[:,1]**2).reshape(100, 200).numpy()
    
    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(amp_map, aspect='auto', origin='lower',
                   extent=[0, 504, 0, 40], cmap='hot')
    ax.axhline(y=20, color='cyan', linestyle='--', alpha=0.7, label='BM1 (z=20)')
    ax.axhline(y=40, color='cyan', linestyle='-', alpha=0.7, label='BM2 (z=40)')
    ax.set_xlabel('x (μm)', fontsize=11)
    ax.set_ylabel('z (μm)', fontsize=11)
    ax.set_title(f'|U|(x, z) Field Map  |  θ=0°, δ=0, w=10  |  epoch {ckpt["epoch"]}',
                 fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='|U|', shrink=0.8)
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()

---
## Cell 5: Red Flag History

In [ ]:
rf_path = EXP_DIR / 'red_flag_history.json'
if rf_path.exists():
    with open(rf_path) as f:
        rf_data = json.load(f)
    
    if rf_data:
        epochs_rf = [d['epoch'] for d in rf_data]
        cov = [d['interior_cov'] for d in rf_data]
        bm1 = [d['bm1_mean_amp'] for d in rf_data]
        bm2 = [d['bm2_mean_amp'] for d in rf_data]
        sens = [d['design_sensitivity'] for d in rf_data]
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        ax = axes[0]
        ax.plot(epochs_rf, cov, 'b-o', markersize=3)
        ax.axhline(y=0.05, color='red', linestyle='--', label='Uniform threshold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Interior CoV')
        ax.set_title('Interior Variation (higher = better)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        ax = axes[1]
        ax.plot(epochs_rf, bm1, 'g-o', markersize=3, label='BM1')
        ax.plot(epochs_rf, bm2, 'r-o', markersize=3, label='BM2')
        ax.axhline(y=0.05, color='gray', linestyle='--', label='Target')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Mean |U| in BM')
        ax.set_title('BM Boundary Learning (lower = better)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        ax = axes[2]
        ax.plot(epochs_rf, sens, 'm-o', markersize=3)
        ax.axhline(y=0.01, color='red', linestyle='--', label='Min threshold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('|U| range (w1 sweep)')
        ax.set_title('Design Variable Sensitivity')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Latest status
        latest = rf_data[-1]
        print(f"Latest check (epoch {latest['epoch']}):")
        print(f"  Red flag: {latest['has_red_flag']}")
        print(f"  Warning: {latest['has_warning']}")
    else:
        print('No red flag checks recorded yet.')
else:
    print('red_flag_history.json not found. Training may still be running.')
    
    # Live check on current model
    if ckpts:
        print('\nRunning live red flag check on latest checkpoint...')
        report = detect_red_flags(model, DEVICE)
        print(report.summary())

---
## Cell 6: Design Variable Response (Sensitivity Check)

In [ ]:
if ckpts:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    N_s = 200
    x_center = torch.full((N_s,), 252.0)  # domain center
    z_opd = torch.zeros(N_s)
    sin_0 = torch.zeros(N_s)
    cos_1 = torch.ones(N_s)
    
    sweeps = [
        ('delta_bm1', 2, torch.linspace(-10, 10, N_s), 'δ₁ sweep [-10, 10]'),
        ('delta_bm2', 3, torch.linspace(-10, 10, N_s), 'δ₂ sweep [-10, 10]'),
        ('w1', 4, torch.linspace(5, 20, N_s), 'w₁ sweep [5, 20]'),
        ('w2', 5, torch.linspace(5, 20, N_s), 'w₂ sweep [5, 20]'),
    ]
    
    for ax, (name, col_idx, sweep_vals, title) in zip(axes.flat, sweeps):
        with torch.no_grad():
            # Default values
            d1 = torch.zeros(N_s)
            d2 = torch.zeros(N_s)
            w1 = torch.full((N_s,), 10.0)
            w2 = torch.full((N_s,), 10.0)
            
            parts = [x_center, z_opd, d1, d2, w1, w2, sin_0, cos_1]
            parts[col_idx] = sweep_vals
            coords = torch.stack(parts, dim=1)
            
            U = model(coords)
            amp = torch.sqrt(U[:,0]**2 + U[:,1]**2).numpy()
        
        ax.plot(sweep_vals.numpy(), amp, 'b-', linewidth=1.5)
        ax.set_xlabel(name + ' (μm)')
        ax.set_ylabel('|U| at OPD center')
        ax.set_title(f'{title}  |  range={amp.max()-amp.min():.4f}')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Design Variable Sensitivity (epoch {ckpt["epoch"]})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Next Steps

- 학습이 수렴되면 → `04_pinn_evaluation.ipynb`에서 최종 검증
- Red flag 발생 시 → 학습 중단, 하이퍼파라미터 조정 후 재시작
- 모든 검증 통과 → Phase D (FNO 증류) 진행